In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip uninstall -y numpy peft accelerate transformers datasets pandas pyarrow -q

!pip install -q \
  "numpy<2.0.0" \
  pandas \
  pyarrow \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  peft==0.7.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

import os
import time
import json
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch import nn
from google.colab import drive
from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Kết nối Drive và cấu hình đường dẫn
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_L14_Baseline_Final"
os.makedirs(SAVE_DIR, exist_ok=True)

# Tối ưu bộ nhớ GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. DATASET & PREPROCESS
dataset = load_dataset("anson-huang/mirage-news")
model_name = "openai/clip-vit-large-patch14"
processor = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": torch.tensor(example["label"], dtype=torch.long)
    }

encoded_dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names)
encoded_dataset.set_format("torch")

# 3. MODEL ARCHITECTURE (FFT + ALIGNMENT)
class CLIPForMFND(nn.Module):
    def __init__(self, base_model, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        self.clip = base_model
        embed_dim = base_model.config.projection_dim # 768 cho L/14

        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, return_dict=True)
        h_T, h_I = outputs.text_embeds, outputs.image_embeds

        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)
            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

base_model = CLIPModel.from_pretrained(model_name)
model = CLIPForMFND(base_model).to(device)

# 4. METRICS & TRAINING ARGS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2, # Giảm để tránh OOM cho bản Large
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 5. EXECUTION & EFFICIENCY MEASUREMENT
def get_param_stats(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, (trainable/total)*100

train_p, all_p, percent_p = get_param_stats(model)
print(f"\n📊 Params: {all_p:,} | Trainable: {train_p:,} ({percent_p:.2f}%)")

print("\n🚀 Training Baseline L/14...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()
trainer.train()
train_time_min = (time.time() - start_train) / 60
peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

# 6. EVALUATION ON 5 TESTS & LATENCY
test_splits = sorted([s for s in dataset.keys() if "test" in s])
results = {}

for split in test_splits:
    enc_test = dataset[split].map(preprocess, remove_columns=dataset[split].column_names)
    enc_test.set_format("torch")

    start_eval = time.time()
    out = trainer.predict(enc_test)
    latency = ((time.time() - start_eval) / len(enc_test)) * 1000

    m = out.metrics
    results[split] = {
        "Acc (%)": round(m["test_accuracy"]*100, 2),
        "F1 (%)": round(m["test_f1"]*100, 2),
        "Latency (ms/sample)": round(latency, 2)
    }

# Tính OOD Average (Test 2-5)
ood = [s for s in test_splits if s != "test1_nyt_mj"]
results["OOD_Average"] = {k: round(np.mean([results[s][k] for s in ood]), 2) for k in results[test_splits[0]]}

results["Hardware_Stats"] = {
    "Train_Time_Min": round(train_time_min, 2),
    "Peak_VRAM_GB": round(peak_vram, 2),
    "Trainable_Params": train_p
}

# 7. SAVE & DISPLAY
df = pd.DataFrame(results).T
print("\n" + "="*60 + "\n", df.to_markdown(), "\n" + "="*60)

with open(os.path.join(SAVE_DIR, "results_L14_Baseline.json"), "w") as f:
    json.dump(results, f, indent=4)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2500 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



📊 Params: 431,160,835 | Trainable: 431,160,835 (100.00%)

🚀 Training Baseline L/14...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.230600,0.024322,0.996800,0.996006,0.997600,0.996803
2,0.041400,0.025561,0.996000,0.999195,0.992800,0.995987
3,0.012700,0.015797,0.997200,0.996803,0.997600,0.997201
4,0.001700,0.018442,0.997600,0.995223,1.000000,0.997606
5,0.001500,0.012511,0.998400,0.996810,1.000000,0.998403



 |                 |   Acc (%) |   F1 (%) |   Latency (ms/sample) |   Train_Time_Min |   Peak_VRAM_GB |   Trainable_Params |
|:----------------|----------:|---------:|----------------------:|-----------------:|---------------:|-------------------:|
| test1_nyt_mj    |     100   |   100    |                 37.86 |           nan    |         nan    |      nan           |
| test2_bbc_dalle |      38.6 |     4.36 |                 34.14 |           nan    |         nan    |      nan           |
| test3_cnn_dalle |      36.4 |     4.22 |                 32.56 |           nan    |         nan    |      nan           |
| test4_bbc_sdxl  |      84.2 |    85.92 |                 32.08 |           nan    |         nan    |      nan           |
| test5_cnn_sdxl  |      83.2 |    85.47 |                 33    |           nan    |         nan    |      nan           |
| OOD_Average     |      60.6 |    44.99 |                 32.94 |           nan    |         nan    |      nan           |
| Hard

In [ ]:
from google.colab import runtime

# Sau khi đã lưu xong results.json và model checkpoint...
print("✅ Mọi tác vụ đã hoàn thành. Đang đóng session để tiết kiệm GPU...")

# Lệnh này sẽ ngắt kết nối và xóa runtime hiện tại
runtime.unassign()

✅ Mọi tác vụ đã hoàn thành. Đang đóng session để tiết kiệm GPU...
